# [1.2] Induction circuits: 현대화된 TransformerLens 실습

**Ubuntu + uv + TransformerLens 3.6 (2026-08)**

이 노트북은 ARENA의 induction-circuit 실습을 2026년의 TransformerLens API와 연구 맥락으로 다시 쓴 버전이다. GPT-2와 2-layer attention-only toy model을 이용해 **관찰 → 정량화 → 직접 기여 → 개입 → weight 가설 → path 검증**의 실험 문법을 연습한다.

> 핵심 위치: induction circuit은 일반 ICL의 완성 이론이 아니라, 완전히 해부 가능한 **token-level associative copying**의 표준 예제다.

## 2026년 관점

고전 induction algorithm은 `[A, B, …, A] → B`이다. 이전 token 정보를 한 위치 뒤에 쓰는 head와, 현재 token과 그 정보를 matching해 continuation을 읽는 head가 결합한다. 이 노트북에서 반복 random token의 후반 loss가 내려가는 현상은 **exact in-context copying / repetition exploitation**의 행동 증거다. 분류·번역·함수 학습 같은 abstractive few-shot ICL 전체의 증거로 읽지 않는다.

현대적 판정 순서는 다음이다.

`pattern metric → OV/direct-logit function → causal intervention → path-specific control → 여러 prompt에서 재현`

따라서 높은 induction score는 `induction-head candidate`를 뜻할 뿐이다. candidate가 정답 logit을 올리는지, 실제 task에 필요한지, 추정한 edge를 통해 효과를 내는지는 별도 실험으로 확인한다.

## 학습 목표와 실행 순서

이 노트북은 모델을 왕복하지 않는다.

1. **toy attention-only model**에서 cache와 작성한 detector를 관찰한다.
2. repeated sequence에서 per-token loss를 구현한다.
3. pattern score를 DLA·ablation·patching과 연결하고 toy circuit을 해부한다.
4. toy와 cache를 해제한 뒤 **GPT-2 Small**에 같은 최소 실험을 한 번 전이한다.

권장 루프는 `shape 예측 → 직접 구현 → reference와 비교 → seed·길이·prompt를 바꿔 반례 찾기`다.

## Ubuntu + uv 환경

GTX 1070은 Pascal(sm_61) GPU다. 2026년 PyPI 기본 PyTorch wheel은 CUDA 13 계열이며 Pascal을 지원하지 않는다. 이 노트북을 GTX 1070에서 GPU로 실행하려면 **CUDA 12.6 PyTorch index**를 명시한다. FP32를 사용한다. Pascal에는 Tensor Core가 없으므로 FP16 이득이 작고, 이 분석은 수치 해석이 중요하다.

```bash
mkdir -p ~/mech-interp && cd ~/mech-interp
uv init --bare --python 3.12
uv add --index pytorch-cu126=https://download.pytorch.org/whl/cu126 torch
uv add "transformer-lens==3.6.0" "circuitsvis==1.43.3" "plotly>=6" \
       "ipykernel>=7" "jupyterlab>=4"
uv run python -m ipykernel install --user --name tl-3-6 --display-name "Python (TransformerLens 3.6)"
uv run jupyter lab
```

JupyterLab에서 **Python (TransformerLens 3.6)** 커널을 선택한다. `uv.lock`은 유지한다. GPU가 없거나 CUDA가 인식되지 않으면 코드가 CPU로 fallback하지만, all-head ablation은 느릴 수 있다.

### 메모리 안전 원칙

- toy와 GPT-2를 동시에 메모리에 올리지 않는다.
- GPT-2 loss에는 logits만, head 탐색에는 pattern hook만 쓴다. 전체 activation cache는 만들지 않는다.
- 반복 forward는 `torch.inference_mode()` 아래에서 실행한다.
- Pylance/extension host와 Jupyter kernel은 별도 프로세스다. Pylance RSS가 계속 증가하면 kernel 정리만으로 줄지 않으므로 extension host reload가 필요하다.

## 공통 setup — import는 이 셀 한 곳에서만

In [ ]:
from __future__ import annotations

import functools
import gc
import inspect
from dataclasses import dataclass
from importlib.metadata import version

import circuitsvis as cv
import numpy as np
import plotly.express as px
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from IPython.display import display
from jaxtyping import Float, Int
from packaging.version import Version
from torch import Tensor
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utilities,
)
from transformer_lens.hook_points import HookPoint
from transformer_lens.model_bridge import TransformerBridge

assert Version(version("transformer-lens")) == Version("3.6.0")
assert Version(version("transformers")) >= Version("5.4.0")
assert {"model_name", "load_weights"} <= set(
    inspect.signature(TransformerBridge.boot_transformers).parameters
)

ModelLike = HookedTransformer | TransformerBridge
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

print(f"Torch={torch.__version__}, TransformerLens={version('transformer-lens')}")
print(f"Transformers={version('transformers')}, device={device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
else:
    print("CPU fallback: keep optional exhaustive experiments disabled.")

In [ ]:
def to_numpy(x: torch.Tensor | np.ndarray) -> np.ndarray:
    return x.detach().float().cpu().numpy() if isinstance(x, torch.Tensor) else x


def model_device(model: ModelLike) -> torch.device:
    return next(model.parameters()).device


def imshow(x, title: str, x_labels=None, y_labels=None, **layout) -> None:
    fig = px.imshow(to_numpy(x), aspect="auto", color_continuous_scale="RdBu")
    fig.update_layout(title=title, **layout)
    if x_labels is not None:
        fig.update_xaxes(
            tickmode="array", tickvals=list(range(len(x_labels))), ticktext=x_labels
        )
    if y_labels is not None:
        fig.update_yaxes(
            tickmode="array", tickvals=list(range(len(y_labels))), ticktext=y_labels
        )
    fig.show()


def line(y, title: str, x=None, **layout) -> None:
    fig = px.line(x=x, y=to_numpy(y), markers=True)
    fig.update_layout(title=title, xaxis_title="position", yaxis_title="value", **layout)
    fig.show()


def correct_log_probs(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    """log p(tokens[:, s+1] | tokens[:, :s+1]) for each prediction position."""
    if logits.ndim != 3 or tokens.ndim != 2:
        raise ValueError("Expected logits [batch, pos, vocab], tokens [batch, pos]")
    return (
        logits[:, :-1]
        .log_softmax(-1)
        .gather(-1, tokens[:, 1:].unsqueeze(-1))
        .squeeze(-1)
    )


def diagonal_mean(pattern: torch.Tensor, offset: int) -> torch.Tensor:
    return pattern.diagonal(offset=offset, dim1=-2, dim2=-1).mean(-1)


def plot_per_token_loss(
    loss: torch.Tensor,
    str_tokens: list[str],
    seq_len: int,
    *,
    title: str,
) -> None:
    if loss.ndim != 1:
        raise ValueError(f"Expected [prediction], got {tuple(loss.shape)}")
    target_tokens = str_tokens[1:]
    if len(target_tokens) != loss.numel():
        raise ValueError("Token labels and per-token losses are misaligned")

    fig = px.line(x=np.arange(loss.numel()), y=to_numpy(loss), markers=True)
    fig.update_traces(
        customdata=np.asarray(target_tokens, dtype=object)[:, None],
        hovertemplate=(
            "prediction=%{x}<br>target=%{customdata[0]}<br>"
            "loss=%{y:.3f}<extra></extra>"
        ),
    )
    fig.add_vline(x=seq_len - 0.5, line_dash="dash", line_color="black")
    fig.update_layout(
        title=title,
        xaxis_title="prediction index",
        yaxis_title="negative log-probability",
    )
    fig.show()

# 1️⃣ Toy model: 관찰과 detector

2-layer attention-only Shortformer에서 실험 문법을 먼저 완결한다. 아래 fuzzy detector 구간은 기존 작업을 보존했으며, import 두 줄만 공통 setup으로 옮겼다.

In [ ]:
TOY_REPO_ID = "callummcdougall/attn_only_2L_half"
TOY_FILENAME = "attn_only_2L_half.pth"

toy_cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,
    positional_embedding_type="shortformer",
    device=device,
)
weights_path = hf_hub_download(repo_id=TOY_REPO_ID, filename=TOY_FILENAME)
toy = HookedTransformer(toy_cfg)
state_dict = torch.load(weights_path, map_location=device, weights_only=True)
toy.load_state_dict(state_dict)
toy.eval()
del state_dict
gc.collect()
print({"layers": toy.cfg.n_layers, "heads": toy.cfg.n_heads, "device": str(model_device(toy))})

## 1.1 작성한 탐색과 fuzzy induction detector

In [18]:
demo_text = ["We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this.",
             "Alice gave the book to Bob, then Carol gave the pen to Dave, then Alice gave the book to",
]


# 해본 결과, 첫 번째 text는 6번 head에 대해 score가 1.8로 제일 높았다. 이때 10번 헤드는 0.95정도 나왔다.
# 두 번째 text는 6번 head에 대해 score 0.22 정도였고, 10번 헤드는 무려 2.5의 점수를 보였다.
# 내 스코어에 attention weight에 대한 sqrt를 적용한다던가 해서 높은 확률에 더 점수를 줄 수도 있으나, 그렇게 하면 metric hacking 이라 하지 않음

In [19]:
demo_tokens = toy.to_tokens(demo_text)
demo_logits, demo_cache = toy.run_with_cache(demo_tokens, remove_batch_dim=False)

In [20]:
print(type(demo_cache))

<class 'transformer_lens.ActivationCache.ActivationCache'>


In [21]:
# pattern 이 키, 0이 몇 번째 Layer 이냐이다
demo_cache['pattern', 0].shape

torch.Size([2, 12, 62, 62])

In [22]:
dir(demo_cache)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_get_cached_ln_scale',
 '_over_ssm_layers',
 '_stack_neuron_results_apply_ln_projected',
 'accumulated_resid',
 'apply_ln_to_stack',
 'apply_slice_to_batch_dim',
 'cache_dict',
 'compute_head_results',
 'compute_ssm_effective_attention',
 'compute_ssm_state',
 'decompose_resid',
 'get_full_resid_decomposition',
 'get_neuron_results',
 'has_batch_dim',
 'has_embed',
 'has_pos_embed',
 'items',
 'keys',
 'logit_attrs',
 'model',
 'remove_batch_dim',
 'ssm_layers',
 'stack_activation',
 'stack_head_results',
 'stack_neuron_results',
 'to',
 'toggle_autodiff',
 'value

In [23]:
#(Batch 가 없으므로) Query: 62토큰, 12헤드, 64차원
embed_dim = demo_cache['k', 0][-2]
demo_cache['k', 0].shape


torch.Size([2, 62, 12, 64])

### Detector를 mechanism 판정기로 오해하지 않기

offset diagonal 평균은 훌륭한 detector다. 하지만 높은 score가 있어도 correct token을 억제하는 anti-induction-like head, redundant head, 다른 task에 무관한 head가 있을 수 있다. 다음 장에서 DLA와 ablation을 붙여 `pattern → function → causal effect`를 검사한다.

또한 token-level diagonal OV score가 낮아도 multi-token lexical / semantic continuation을 운반하는 head일 수 있다. 이 노트북의 OV test는 exact token copying에 한정된 probe다.

### Improved fuzzy induction-head score

아래 구현은 원래 아이디어인 `현재 query token`과 `attended source의 직전 token` 사이의 embedding cosine similarity를 보존한다. 다만 기존 구현과 알고리즘적으로 다음이 다르다.

- **argmax 하나 → 전체 attention mass:** 의미상 대응하는 source가 2순위이거나 여러 위치에 분산되어도 반영한다.
- **`source <= query` → `source < query`:** 현재 위치에 attend한 뒤 직전 bigram의 유사도를 재는 자기참조 false positive를 제거한다. BOS(`source=0`)도 직전 token이 없으므로 제외한다.
- **문장 전체 합 → opportunity-conditioned mean:** fuzzy/exact prefix 후보가 실제로 존재하는 query에서 평균내므로 문장 길이만으로 점수가 커지지 않는다. 기회 수를 함께 반환해 작은 표본도 드러낸다.
- **raw cosine → vocabulary-mean-centered cosine의 threshold 초과분:** embedding 공간의 공통 방향에 의한 양의 cosine bias를 줄인다. exact match와 semantic-only match를 별도로 보고하여 fuzzy 점수가 exact repetition에 가려지는 것도 막는다.
- **고정 layer·묵시적 shape → 명시적 인자와 검증:** 잘못된 batch/cache shape를 조용히 broadcast하지 않고 오류로 처리한다.

`semantic_threshold`는 결과를 보고 조정하는 값이 아니다. 아래 `0.35`는 이 toy exploration의 명시적 예시값이며, 연구용 비교에서는 별도의 synonym/control validation set에서 먼저 고정해야 한다. 이 점수는 여전히 **attention-pattern candidate score**이므로 OV/DLA와 ablation 없이 기능적·인과적 induction을 주장하지 않는다.


In [24]:
# + now with batch dimension
demo_tokens = toy.to_tokens(demo_text)
demo_logits, demo_cache = toy.run_with_cache(demo_tokens, remove_batch_dim=False)


In [25]:
ebd =  demo_cache["embed"]

In [26]:
TOKEN_PADDING = 1

In [27]:
@dataclass(frozen=True)
class InductionHeadScoreResult:
    # All score tensors have shape [n_heads] and values in [0, 1].
    combined: torch.Tensor
    exact: torch.Tensor
    semantic_only: torch.Tensor
    combined_opportunities: int
    exact_opportunities: int
    semantic_opportunities: int
    per_query_combined: torch.Tensor
    combined_opportunity_mask: torch.Tensor


@torch.inference_mode()
def fuzzy_induction_head_scores(
    model: HookedTransformer,
    cache: ActivationCache,
    tokens: torch.Tensor,
    *,
    layer: int,
    semantic_threshold: float,
    require_observed_next_token: bool = True,
) -> InductionHeadScoreResult:
    """Score exact and fuzzy prefix-matching attention for every head in one layer.

    For query q and source k, induction-style prefix evidence compares x[q] with x[k-1].
    Only 1 <= k < q is valid: k=0 has no predecessor, and k=q is self-attention rather
    than retrieval of an earlier continuation. All matching attention mass is retained.

    Semantic evidence is the linearly rescaled excess of centered cosine similarity over
    semantic_threshold. Exact-token matches are scored separately and excluded from the
    semantic-only channel. Scores are averaged only over queries that contain at least one
    eligible match, which removes raw sequence-length dependence.
    *결국 봐야할 건 'Extact Match' 가 있을 때 모델이 얼마나 가까운 Embedding을 고르느냐' 이기 때문*

    """
    if not 0.0 <= semantic_threshold < 1.0:
        raise ValueError("semantic_threshold must satisfy 0 <= threshold < 1")

    if tokens.ndim == 1:
        tokens = tokens[None, :]
    if tokens.ndim != 2:
        raise ValueError(f"Expected tokens with shape [seq] or [batch, seq], got {tuple(tokens.shape)}")

    pattern = cache["pattern", layer]
    token_embeddings = cache["embed"]
    if pattern.ndim == 3:
        pattern = pattern.unsqueeze(0)
    if token_embeddings.ndim == 2:
        token_embeddings = token_embeddings.unsqueeze(0)
    if pattern.ndim != 4 or token_embeddings.ndim != 3:
        raise ValueError(
            "Expected pattern [batch, head, q, k] and embed [batch, seq, d_model]"
        )
    batch_size, max_seq_len = tokens.shape
    if max_seq_len < 2:
        raise ValueError("At least two tokens are required")
    if (
        pattern.shape[0] != batch_size
        or pattern.shape[-2:] != (max_seq_len, max_seq_len)
        or token_embeddings.shape[:2] != (batch_size, max_seq_len)
    ):
        raise ValueError("tokens, attention pattern, and cached embeddings disagree on sequence length")

    # Center by the whole vocabulary rather than by this prompt, then normalize once.
    # This preserves the original cosine-similarity idea while reducing embedding anisotropy.
    vocabulary_center = model.W_E.detach().mean(dim=0)
    normalized_embeddings = F.normalize(token_embeddings - vocabulary_center, dim=-1)

    # prefix_cosine[q, k] = cos(E[x[q]], E[x[k-1]]); column 0 has no predecessor.
    prefix_cosine = token_embeddings.new_zeros((batch_size, max_seq_len, max_seq_len))
    prefix_cosine[:, :, 1:] = torch.einsum(
        "bqd,bkd->bqk", normalized_embeddings, normalized_embeddings[:, :-1]
    )
    valid_tokens = tokens.ne(TOKEN_PADDING)
    valid_query_key_pairs = valid_tokens[:, :, None] & valid_tokens[:, None, :]
    prefix_cosine.masked_fill_(~valid_query_key_pairs, 0.0)

    exact_prefix = torch.zeros(
        (batch_size, max_seq_len, max_seq_len), dtype=torch.bool, device=tokens.device
    )
    exact_prefix[:, :, 1:] = tokens[:, :, None].eq(tokens[:, None, :-1])
    exact_prefix &= valid_query_key_pairs

    query_index = torch.arange(max_seq_len, device=tokens.device)[None, :, None]
    source_index = torch.arange(max_seq_len, device=tokens.device)[None, None, :]
    strict_past = (source_index >= 1) & (source_index < query_index)

    exact_evidence = (exact_prefix & strict_past).to(pattern.dtype)
    #여기까지. causality를 고려한(자기 이전 토큰만 Query가 고려가능), Exact match
    semantic_evidence = (
        (prefix_cosine - semantic_threshold) / (1.0 - semantic_threshold)
    ).clamp(0.0, 1.0)
    semantic_evidence = semantic_evidence * strict_past * (~exact_prefix)
    combined_evidence = exact_evidence + semantic_evidence
    #여기까지. 딱 안맞는것도 가중치 + Token별로 가중치 자름
    def opportunity_conditioned_mean(evidence: torch.Tensor):
        per_query = torch.einsum("bhqk,bqk->bhq", pattern, evidence)
        opportunity_mask = evidence.gt(0).any(dim=-1)
        if require_observed_next_token:
            positions = torch.arange(max_seq_len, device=tokens.device)[None, :]
            last_valid_index = positions.masked_fill(~valid_tokens, -1).amax(dim=1)
            opportunity_mask &= positions < last_valid_index[:, None]
        if not opportunity_mask.any():
            return pattern.new_zeros(pattern.shape[1]), per_query, opportunity_mask
        scores = per_query.permute(1, 0, 2)[:, opportunity_mask].mean(dim=-1)
        return scores, per_query, opportunity_mask

    combined, per_query_combined, combined_mask = opportunity_conditioned_mean(combined_evidence)
    exact, _, exact_mask = opportunity_conditioned_mean(exact_evidence)
    semantic_only, _, semantic_mask = opportunity_conditioned_mean(semantic_evidence)

    return InductionHeadScoreResult(
        combined=combined,
        exact=exact,
        semantic_only=semantic_only,
        combined_opportunities=int(combined_mask.sum().item()),
        exact_opportunities=int(exact_mask.sum().item()),
        semantic_opportunities=int(semantic_mask.sum().item()),
        per_query_combined=per_query_combined,
        combined_opportunity_mask=combined_mask,
    )


improved_scores = fuzzy_induction_head_scores(
    toy,
    demo_cache,
    demo_tokens,
    layer=1,
    semantic_threshold=0.35,  # Calibrate on held-out synonym/control pairs for real studies.
)
print(
    f"Opportunities: combined={improved_scores.combined_opportunities}, "
    f"exact={improved_scores.exact_opportunities}, "
    f"semantic-only={improved_scores.semantic_opportunities}"
)
for head, (combined, exact, semantic) in enumerate(
    zip(improved_scores.combined, improved_scores.exact, improved_scores.semantic_only)
):
    print(
        f"L1H{head:02d}: combined={combined.item():.4f}, "
        f"exact={exact.item():.4f}, semantic-only={semantic.item():.4f}"
    )
print(f"Top fuzzy induction-pattern candidate: L1H{improved_scores.combined.argmax().item()}")

Opportunities: combined=28, exact=19, semantic-only=10
L1H00: combined=0.0516, exact=0.0720, semantic-only=0.0075
L1H01: combined=0.0562, exact=0.0775, semantic-only=0.0101
L1H02: combined=0.0420, exact=0.0429, semantic-only=0.0362
L1H03: combined=0.0728, exact=0.1014, semantic-only=0.0110
L1H04: combined=0.1089, exact=0.1501, semantic-only=0.0196
L1H05: combined=0.0525, exact=0.0721, semantic-only=0.0102
L1H06: combined=0.0467, exact=0.0601, semantic-only=0.0165
L1H07: combined=0.0471, exact=0.0597, semantic-only=0.0185
L1H08: combined=0.0291, exact=0.0274, semantic-only=0.0294
L1H09: combined=0.0617, exact=0.0817, semantic-only=0.0176
L1H10: combined=0.1952, exact=0.2551, semantic-only=0.0618
L1H11: combined=0.0340, exact=0.0493, semantic-only=0.0016
Top fuzzy induction-pattern candidate: L1H10


### 여기까지의 결론

Fuzzy score는 “유사한 prefix가 있는 과거 위치에 attention mass를 두는가?”를 측정한다. source의 OV가 정답 token을 쓰는지, head가 task에 필요한지는 아직 말하지 않는다. 다음부터 controlled input에서 `behavior → pattern → function → causality`를 분리한다.

# 2️⃣ Exercise — plot per-token loss on repeated sequence

입력은 `[BOS, x₀, …, xₙ₋₁, x₀, …, xₙ₋₁]`이다. logits position `s`는 token `s+1`을 예측하므로 loss 길이는 input보다 하나 짧다.

실행 전에 생성 shape, loss index `seq_len`의 target, loss가 내려갈 구간을 직접 적는다. 아래는 실행 가능한 reference implementation이며, 먼저 scratch cell에서 docstring만 보고 구현해 보는 것을 권한다.

In [ ]:
def generate_repeated_tokens(
    model: ModelLike,
    seq_len: int,
    batch_size: int = 1,
    *,
    seed: int = 0,
) -> Int[Tensor, "batch full_seq_len"]:
    """Return [BOS, random_half, random_half], shape [batch, 1 + 2*seq_len]."""
    if seq_len < 2 or batch_size < 1:
        raise ValueError("seq_len must be >= 2 and batch_size must be positive")

    bos_token_id = model.tokenizer.bos_token_id
    if bos_token_id is None:
        bos_token_id = model.tokenizer.eos_token_id
    if bos_token_id is None:
        raise ValueError("Tokenizer has neither BOS nor EOS")

    # Local generator preserves notebook-global RNG state.
    generator = torch.Generator(device="cpu").manual_seed(seed)
    random_half = torch.randint(
        0,
        model.cfg.d_vocab,
        (batch_size, seq_len),
        generator=generator,
        dtype=torch.long,
    )
    prefix = torch.full((batch_size, 1), bos_token_id, dtype=torch.long)
    return torch.cat((prefix, random_half, random_half), -1).to(model_device(model))


@torch.inference_mode()
def run_and_cache_model_repeated_tokens(
    model: ModelLike,
    seq_len: int,
    batch_size: int = 1,
    *,
    seed: int = 0,
) -> tuple[Tensor, Tensor, ActivationCache]:
    repeated_tokens = generate_repeated_tokens(
        model, seq_len, batch_size, seed=seed
    )
    repeated_logits, repeated_cache = model.run_with_cache(
        repeated_tokens,
        return_type="logits",
    )
    return repeated_tokens, repeated_logits, repeated_cache

In [ ]:
SEQ_LEN = 50
rep_tokens, rep_logits, rep_cache = run_and_cache_model_repeated_tokens(
    toy, SEQ_LEN, batch_size=1
)

assert rep_tokens.shape == (1, 1 + 2 * SEQ_LEN)
assert torch.equal(rep_tokens[:, 1 : 1 + SEQ_LEN], rep_tokens[:, 1 + SEQ_LEN :])
assert rep_logits.shape[:2] == rep_tokens.shape

rep_loss = -correct_log_probs(rep_logits, rep_tokens)[0]
rep_str_tokens = toy.to_str_tokens(rep_tokens[0])
assert rep_loss.shape == (2 * SEQ_LEN,)
assert torch.isfinite(rep_loss).all()

first_loss = rep_loss[:SEQ_LEN].mean()
second_loss = rep_loss[SEQ_LEN:].mean()
print(f"First-copy mean loss : {first_loss.item():.3f}")
print(f"Second-copy mean loss: {second_loss.item():.3f}")
print(f"Improvement: {(first_loss - second_loss).item():.3f}")
plot_per_token_loss(
    rep_loss,
    rep_str_tokens,
    SEQ_LEN,
    title="Toy model per-token loss on repeated random tokens",
)

# Later toy sections assume the only batch dimension has been removed.
rep_cache.remove_batch_dim()

### 해석 checkpoint

- y축은 negative log-probability이므로 낮을수록 좋다.
- loss `0…seq_len-1`은 첫 copy, `seq_len…2*seq_len-1`은 둘째 copy를 예측한다.
- 둘째 copy의 첫 token은 `xₙ₋₁ → x₀` transition을 본 적이 없어 causal metric에서는 보통 제외한다.
- 확장 실습: seed 3개와 `SEQ_LEN=10, 50, 100`에서 improvement를 비교한다.

## 2.1 Exact induction stripe

두 번째 `xᵢ` query가 첫 번째 `xᵢ` 바로 다음 위치를 읽으면 attention matrix offset `1 - seq_len`에 stripe가 생긴다.

In [ ]:
def exact_induction_scores_from_cache(
    cache: ActivationCache,
    n_layers: int,
    seq_len: int,
) -> Float[Tensor, "layer head"]:
    scores = []
    for layer in range(n_layers):
        pattern = cache["pattern", layer]
        if pattern.ndim != 3:
            raise ValueError("Expected unbatched [head, dest, source] pattern")
        scores.append(diagonal_mean(pattern, 1 - seq_len))
    return torch.stack(scores)


exact_scores = exact_induction_scores_from_cache(rep_cache, toy.cfg.n_layers, SEQ_LEN)
imshow(
    exact_scores,
    "Exact induction-pattern score",
    x_labels=[str(h) for h in range(toy.cfg.n_heads)],
    y_labels=[str(l) for l in range(toy.cfg.n_layers)],
)
flat_top = exact_scores.argmax()
print(
    "Top exact candidate:",
    f"L{(flat_top // toy.cfg.n_heads).item()}H{(flat_top % toy.cfg.n_heads).item()}",
)

for layer in range(toy.cfg.n_layers):
    display(cv.attention.attention_patterns(
        tokens=rep_str_tokens,
        attention=rep_cache["pattern", layer],
        attention_head_names=[f"L{layer}H{h}" for h in range(toy.cfg.n_heads)],
    ))

# 3️⃣ Function과 causality

DLA는 component가 정답 logit에 직접 쓴 양이다. downstream 변화나 necessity는 측정하지 않으므로 ablation·patching과 구분한다.

In [ ]:
def logit_attribution(embed, layer0_result, layer1_result, W_U, tokens):
    correct_unembed = W_U[:, tokens[1:]]
    direct = torch.einsum("sd,ds->s", embed[:-1], correct_unembed)
    layer0 = torch.einsum("shd,ds->sh", layer0_result[:-1], correct_unembed)
    layer1 = torch.einsum("shd,ds->sh", layer1_result[:-1], correct_unembed)
    return torch.cat((direct[:, None], layer0, layer1), -1)


dla = logit_attribution(
    rep_cache["embed"],
    rep_cache["result", 0],
    rep_cache["result", 1],
    toy.W_U,
    rep_tokens[0],
)
correct_logits = rep_logits[0, :-1].gather(
    -1, rep_tokens[0, 1:].unsqueeze(-1)
).squeeze(-1)
torch.testing.assert_close(dla.sum(-1), correct_logits, atol=1e-3, rtol=0)

labels = ["direct"] + [f"L0H{h}" for h in range(12)] + [f"L1H{h}" for h in range(12)]
fig = px.bar(x=labels, y=to_numpy(dla[-(SEQ_LEN - 1):].mean(0)))
fig.update_layout(title="Mean direct correct-token attribution on copy 2")
fig.show()
del fig

In [ ]:
@torch.inference_mode()
def score_induction_heads_with_hooks(
    model: ModelLike,
    tokens: torch.Tensor,
    seq_len: int,
) -> Float[Tensor, "layer head"]:
    store = torch.zeros(
        (model.cfg.n_layers, model.cfg.n_heads),
        device=model_device(model),
    )

    def save_score(pattern: torch.Tensor, hook: HookPoint) -> None:
        store[hook.layer()] = diagonal_mean(pattern, 1 - seq_len).mean(0)

    model.reset_hooks()
    model.run_with_hooks(
        tokens,
        return_type=None,
        fwd_hooks=[
            (utilities.get_act_name("pattern", layer), save_score)
            for layer in range(model.cfg.n_layers)
        ],
    )
    return store


hook_scores = score_induction_heads_with_hooks(toy, rep_tokens, SEQ_LEN)
torch.testing.assert_close(hook_scores, exact_scores)
print("Hook and cache scores agree.")

## Ablation

Zero ablation은 necessity를 묻지만 OOD intervention일 수 있다. 먼저 L1H4 하나를 확인하고, 전체 24-head sweep은 RAM·시간을 고려해 flag 아래 둔다.

In [ ]:
def repeated_second_half_loss(logits, tokens):
    seq_len = (tokens.shape[1] - 1) // 2
    return -correct_log_probs(logits, tokens)[:, -(seq_len - 1):].mean()


def zero_head_hook(z, hook, *, head: int):
    z[:, :, head, :] = 0.0
    return z


@torch.inference_mode()
def ablate_head(model, tokens, *, layer: int, head: int) -> float:
    baseline = repeated_second_half_loss(model(tokens), tokens)
    logits = model.run_with_hooks(
        tokens,
        return_type="logits",
        fwd_hooks=[(
            utilities.get_act_name("z", layer),
            functools.partial(zero_head_hook, head=head),
        )],
    )
    return (repeated_second_half_loss(logits, tokens) - baseline).item()


print("L1H4 ablation Δloss:", round(ablate_head(toy, rep_tokens, layer=1, head=4), 3))

RUN_EXHAUSTIVE_ABLATION = False
if RUN_EXHAUSTIVE_ABLATION:
    ablation_scores = torch.empty((toy.cfg.n_layers, toy.cfg.n_heads), device=device)
    for layer in range(toy.cfg.n_layers):
        for head in range(toy.cfg.n_heads):
            ablation_scores[layer, head] = ablate_head(
                toy, rep_tokens, layer=layer, head=head
            )
    imshow(ablation_scores, "Zero-ablation Δloss")
else:
    print("Set RUN_EXHAUSTIVE_ABLATION=True for the 24-forward sweep.")

### Clean / corrupted activation patching

Ablation asks whether a component is necessary under one intervention. Patching asks whether information from a clean run is sufficient to restore a chosen behavior in a corrupted run. Always define a task metric and include controls (random same-layer head, matched position, reverse patch). The example below patches one induction head's **key activation**, which is more specific than ablating the whole head but is still not a complete edge-level proof.

In [ ]:
def patch_one_head(clean_activation: torch.Tensor, head: int):
    def patch_hook(activation: torch.Tensor, hook):
        activation[:, :, head, :] = clean_activation[:, :, head, :]
        return activation
    return patch_hook


clean_tokens = generate_repeated_tokens(toy, SEQ_LEN)
corrupted_tokens = clean_tokens.clone()
# Break one earlier token identity while preserving sequence length and all later positions.
corrupted_tokens[:, 8] = (corrupted_tokens[:, 8] + 1) % toy.cfg.d_vocab

clean_logits, clean_cache = toy.run_with_cache(clean_tokens, names_filter=lambda n: n == utilities.get_act_name("k", 1))
corrupted_logits = toy(corrupted_tokens)
patched_logits = toy.run_with_hooks(
    corrupted_tokens,
    fwd_hooks=[(utilities.get_act_name("k", 1), patch_one_head(clean_cache["k", 1], head=4))],
)

position = SEQ_LEN + 8  # this logit predicts the following repeated token
correct_token = clean_tokens[0, position + 1].item()
def correct_logit(logits): return logits[0, position, correct_token].item()
print({"clean_correct_logit": correct_logit(clean_logits), "corrupted_correct_logit": correct_logit(corrupted_logits), "patched_key_correct_logit": correct_logit(patched_logits)})
# Interpret only relative to explicit controls and many random draws, not this single prompt.

# 4️⃣ Reverse-engineering the induction circuit

이 toy model에서는 weight product가 강력하다. attention-only, no LayerNorm, no MLP라는 조건 덕분에 `W_E W_V W_O W_U`와 같은 선형식을 실제 계산과 직접 연결할 수 있다.

일반 LLM에서는 pre-norm, MLP, gating, RoPE, GQA, polysemantic head 때문에 raw `W_Q W_Kᵀ`가 완전한 effective circuit이 아니다. 대형 모델의 weight product는 **가능한 선형 path** 가설이지 사용 증거가 아니다.

In [ ]:
# FactoredMatrix avoids materializing a vocab × vocab matrix (~50k² entries).
A = torch.randn(5, 2, device=device)
B = torch.randn(2, 5, device=device)
AB = FactoredMatrix(A, B)
print("shape / factor dimensions:", AB.shape, AB.ldim, AB.mdim, AB.rdim)
print("factorized norm agrees with dense norm:", AB.norm().item(), (A @ B).norm().item())


def full_ov_circuit(model: HookedTransformer, layer: int, head: int) -> FactoredMatrix:
    # source token → attended token's output-logit effect
    return FactoredMatrix(
        model.W_E @ model.W_V[layer, head],
        model.W_O[layer, head] @ model.W_U,
    )


def top1_copy_accuracy(circuit: FactoredMatrix, vocab_size: int, batch_size: int = 128) -> float:
    indices = torch.randint(0, vocab_size, (batch_size,), device=device)
    sampled_rows = circuit[indices].AB  # [sampled source token, candidate output token]
    return (sampled_rows.argmax(dim=-1) == indices).float().mean().item()

ov_14 = full_ov_circuit(toy, layer=1, head=4)
ov_110 = full_ov_circuit(toy, layer=1, head=10)
print("L1H4 top-1 exact-copy accuracy:", top1_copy_accuracy(ov_14, toy.cfg.d_vocab))
print("L1H10 top-1 exact-copy accuracy:", top1_copy_accuracy(ov_110, toy.cfg.d_vocab))

# Two heads can jointly implement a more functional circuit than either head alone.
effective_ov = FactoredMatrix(
    toy.W_E @ torch.cat((toy.W_V[1, 4], toy.W_V[1, 10]), dim=-1),
    torch.cat((toy.W_O[1, 4] @ toy.W_U, toy.W_O[1, 10] @ toy.W_U), dim=0),
)
print("combined top-1 exact-copy accuracy:", top1_copy_accuracy(effective_ov, toy.cfg.d_vocab))

In [ ]:
# Previous-token position preference: this Shortformer toy model has additive W_pos.
prev_head = 7
W_QK_prev = toy.W_Q[0, prev_head] @ toy.W_K[0, prev_head].T
pos_scores = toy.W_pos @ W_QK_prev @ toy.W_pos.T / toy.cfg.d_head**0.5
mask = torch.tril(torch.ones_like(pos_scores), diagonal=0).bool()
pos_pattern = torch.where(mask, pos_scores, torch.finfo(pos_scores.dtype).min).softmax(-1)
print("mean attention to previous position:", pos_pattern.diag(-1).mean().item())
imshow(pos_pattern[:128, :128], "L0H7 positional attention pattern (Shortformer only)")

# Do not carry this additive token/position decomposition unchanged to RoPE models:
# q_i = R_i W_Q x_i, k_j = R_j W_K x_j, so position is encoded in R_i^T R_j instead.

In [ ]:
def decompose_layer1_input(cache) -> torch.Tensor:
    """[embed, pos_embed, each L0 head result] at each position; batch dimension removed."""
    l0_results = cache["result", 0].permute(1, 0, 2)  # [head, position, d_model]
    return torch.cat((cache["embed"][None], cache["pos_embed"][None], l0_results), dim=0)


def decompose_q_or_k(components: torch.Tensor, model: HookedTransformer, head: int, which: str) -> torch.Tensor:
    W = model.W_Q[1, head] if which == "q" else model.W_K[1, head]
    return torch.einsum("cpm,md->cpd", components, W)


def decompose_attention_scores(q_parts: torch.Tensor, k_parts: torch.Tensor) -> torch.Tensor:
    return torch.einsum("aqd,bkd->abqk", q_parts, k_parts) / toy.cfg.d_head**0.5

single_cache = rep_cache
components = decompose_layer1_input(single_cache)
q_parts = decompose_q_or_k(components, toy, head=4, which="q")
k_parts = decompose_q_or_k(components, toy, head=4, which="k")
torch.testing.assert_close(q_parts.sum(0), single_cache["q", 1][:, 4], rtol=2e-2, atol=1e-3)
torch.testing.assert_close(k_parts.sum(0), single_cache["k", 1][:, 4], rtol=2e-2, atol=1e-3)
labels = ["embed", "pos_embed"] + [f"0.{h}" for h in range(toy.cfg.n_heads)]
imshow(q_parts.pow(2).sum(-1), "L1H4 query-component norms", y_labels=labels)
imshow(k_parts.pow(2).sum(-1), "L1H4 key-component norms", y_labels=labels)

In [ ]:
def k_composition_circuit(model: HookedTransformer, prev_head: int, induction_head: int) -> FactoredMatrix:
    """Current token → previous-token head write → L1 induction-head key match."""
    left = model.W_E @ model.W_Q[1, induction_head]
    right = (
        model.W_K[1, induction_head].T
        @ model.W_O[0, prev_head].T
        @ model.W_V[0, prev_head].T
        @ model.W_E.T
    )
    return FactoredMatrix(left, right)


k_comp = k_composition_circuit(toy, prev_head=7, induction_head=4)
print("K-composition top-1 same-token score:", top1_copy_accuracy(k_comp.T, toy.cfg.d_vocab))

def composition_score(W_A: torch.Tensor, W_B: torch.Tensor) -> float:
    return (W_A @ W_B).norm().div(W_A.norm() * W_B.norm()).item()

W_QK = toy.W_Q @ toy.W_K.transpose(-1, -2)
W_OV = toy.W_V @ toy.W_O
composition = {name: torch.empty((toy.cfg.n_heads, toy.cfg.n_heads), device=device) for name in ["Q", "K", "V"]}
for source_head in range(toy.cfg.n_heads):
    for destination_head in range(toy.cfg.n_heads):
        composition["Q"][source_head, destination_head] = composition_score(W_OV[0, source_head], W_QK[1, destination_head])
        composition["K"][source_head, destination_head] = composition_score(W_OV[0, source_head], W_QK[1, destination_head].T)
        composition["V"][source_head, destination_head] = composition_score(W_OV[0, source_head], W_OV[1, destination_head])
imshow(composition["K"], "Potential K-composition: L0 write → L1 read")

# This measures subspace alignment (potential connectivity), not task-relevant information flow.

In [ ]:
def batched_composition_scores(W_As: FactoredMatrix, W_Bs: FactoredMatrix) -> torch.Tensor:
    """All pair scores without materializing d_model × d_model products."""
    products = W_As[:, None] @ W_Bs[None, :]
    return products.norm() / (W_As.norm()[:, None] * W_Bs.norm()[None, :])

factored_qk = FactoredMatrix(toy.W_Q, toy.W_K.transpose(-1, -2))
factored_ov = FactoredMatrix(toy.W_V, toy.W_O)
torch.testing.assert_close(batched_composition_scores(factored_ov[0], factored_qk[1].T), composition["K"])
print("Batched FactoredMatrix composition agrees with the explicit loop.")

# Avoid `full_circuit.AB` for a full vocab × vocab circuit: it can exceed 8GB by itself.

In [ ]:
def induction_score_after_prev_head_ablation(prev_head: int | None, induction_head: int = 4) -> float:
    observed = {}
    def ablate_z(z, hook):
        if prev_head is not None:
            z[:, :, prev_head, :] = 0.0
        return z
    def read_pattern(pattern, hook):
        observed["score"] = diagonal_mean(pattern[:, induction_head], -(SEQ_LEN - 1)).mean().item()
    toy.run_with_hooks(
        rep_tokens,
        fwd_hooks=[
            (utilities.get_act_name("z", 0), ablate_z),
            (utilities.get_act_name("pattern", 1), read_pattern),
        ],
    )
    return observed["score"]

baseline = induction_score_after_prev_head_ablation(None)
changes = torch.tensor([induction_score_after_prev_head_ablation(h) - baseline for h in range(toy.cfg.n_heads)])
print("baseline L1H4 induction score:", baseline)
print("change after each L0-head ablation:", changes.tolist())
line(changes, "Effect of ablating each L0 head on L1H4's induction pattern")

# This is stronger than a correlational stripe, but still removes all L0-head output paths.
# The next refinement is to patch only the contribution of L0H7 to L1H4's key input,
# then compare Q/K/V and matched-position controls.

# 5️⃣ GPT-2 Small로 한 번만 전이

Toy 실습을 끝낸 뒤 toy·cache·circuit reference를 해제하고 GPT-2를 로드한다.

In [ ]:
for name in [
    "toy", "demo_logits", "demo_cache", "rep_logits", "rep_cache", "rep_loss",
    "dla", "clean_logits", "clean_cache", "corrupted_logits", "patched_logits",
    "single_cache", "components", "q_parts", "k_parts", "ov_14", "ov_110",
    "effective_ov", "k_comp", "W_QK", "W_OV", "composition", "factored_qk",
    "factored_ov", "AB", "A", "B", "_",
]:
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(
        "CUDA allocated/reserved MiB:",
        round(torch.cuda.memory_allocated() / 2**20, 1),
        round(torch.cuda.memory_reserved() / 2**20, 1),
    )
else:
    print("Toy references released.")

## GPT-2 load

`TransformerBridge.boot_transformers`의 raw HF weights를 쓴다. behavior와 pattern만 비교하므로 compatibility mode는 필요 없다. 그 mode는 LayerNorm을 단순 제거하는 것이 아니라 fold/centering으로 legacy 좌표계에 맞추는 기능이다.

In [ ]:
GPT2_ID = "openai-community/gpt2"
gpt2 = TransformerBridge.boot_transformers(
    GPT2_ID,
    device=device,
    dtype=torch.float32,
)
gpt2.eval()
print(type(gpt2).__name__)
print({
    name: getattr(gpt2.cfg, name)
    for name in ["n_layers", "n_heads", "d_model", "d_vocab", "n_ctx"]
})

## 같은 per-token loss — cache 없이

In [ ]:
GPT2_SEQ_LEN = 50
gpt2_tokens = generate_repeated_tokens(gpt2, GPT2_SEQ_LEN, seed=0)
with torch.inference_mode():
    gpt2_logits = gpt2(gpt2_tokens, return_type="logits")

gpt2_loss = -correct_log_probs(gpt2_logits, gpt2_tokens)[0]
print(
    "GPT-2 first/second mean loss:",
    round(gpt2_loss[:GPT2_SEQ_LEN].mean().item(), 3),
    round(gpt2_loss[GPT2_SEQ_LEN:].mean().item(), 3),
)
plot_per_token_loss(
    gpt2_loss,
    gpt2.to_str_tokens(gpt2_tokens[0]),
    GPT2_SEQ_LEN,
    title="GPT-2 per-token loss on repeated random tokens",
)
del gpt2_logits
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Pattern hook으로 GPT-2 candidate 찾기

전체 activation cache 대신 각 pattern이 생성될 때 layer×head score만 저장한다.

In [ ]:
gpt2_score_tokens = generate_repeated_tokens(
    gpt2, GPT2_SEQ_LEN, batch_size=4, seed=1
)
gpt2_scores = score_induction_heads_with_hooks(
    gpt2, gpt2_score_tokens, GPT2_SEQ_LEN
)
imshow(
    gpt2_scores,
    "GPT-2 exact induction-pattern score",
    x_labels=[str(h) for h in range(gpt2.cfg.n_heads)],
    y_labels=[str(l) for l in range(gpt2.cfg.n_layers)],
)
values, indices = gpt2_scores.flatten().topk(8)
print(*[
    {
        "head": f"L{i.item() // gpt2.cfg.n_heads}H{i.item() % gpt2.cfg.n_heads}",
        "score": round(v.item(), 3),
    }
    for v, i in zip(values, indices)
], sep="\n")

# 5️⃣ What this circuit does—and does not—establish

이 toy model에서는 `L0 previous-token write → L1 key match → L1 copying OV → correct logit`이라는 설명이 pattern, activation decomposition, weight product, ablation으로 함께 지지된다. 그래서 **exact token induction**에 대한 강한 mechanistic example이다.

하지만 큰 pre-norm RoPE/GQA 모델에서는 다음을 추가로 검사해야 한다.

- high prefix score가 아니라 task-specific correct-token promotion / suppression
- clean↔corrupted restoration, reverse patch, random-head·matched-position negative control
- 여러 prompt와 seed에서의 재현성
- head-level graph의 polysemanticity를 줄이는 feature/SAE-level 분석
- 큰 graph에서는 EAP/AtP류를 후보 탐색용으로 쓰고, 핵심 edge는 실제 patching으로 재검증

추천 확장 실습: (A) 모든 high-score head를 pattern·DLA·ablation 세 값으로 분류, (B) exact repetition과 multi-token/semantic copying 비교, (C) answer가 context에 없는 abstractive few-shot task를 별도 평가.

## 참고 자료

- TransformerLens 3 migration: https://transformerlensorg.github.io/TransformerLens/content/migrating_to_v3.html
- TransformerBridge model structure: https://transformerlensorg.github.io/TransformerLens/content/model_structure.html
- Olsson et al. (2022), *In-context Learning and Induction Heads*: https://arxiv.org/abs/2209.11895
- Sahin et al. (2025), *In-Context Learning Without Copying*: https://arxiv.org/abs/2511.05743
- Mohammed & Belz (2026), *What Matters More for ICL under Matched Compute Budgets?*: https://openreview.net/forum?id=c01qNs6Ew7
- uv + PyTorch index configuration: https://docs.astral.sh/uv/guides/integration/pytorch/
- PyTorch 2.13 release: CUDA 13 is the default while CUDA 12.6 is the legacy Pascal-compatible route: https://pytorch.org/blog/pytorch-2-13-release-blog/

> 마지막 원칙: 좋은 mechanistic explanation은 예쁜 attention 그림이 아니라, 사전에 정한 metric·통제군·개입에서 새로운 결과를 예측하고 재현하는 설명이다.